In [1]:
from pyspark.sql import SparkSession

# Initialise Spark session
spark = (
    SparkSession.builder.appName("Rideshare_Analysis")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "10g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)


your 131072x1 screen size is bogus. expect trouble
24/08/26 16:47:39 WARN Utils: Your hostname, LAPTOP-354CCOC2 resolves to a loopback address: 127.0.1.1; using 172.21.14.166 instead (on interface eth0)
24/08/26 16:47:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/26 16:47:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


This next section will work on cleaning/preprocessing the 'FHVHV' dataset

In [2]:
# Load FHVHV data
fhvhv_df = spark.read.parquet("../data/raw")

In [3]:
# Examine the initial shape of the dataset

# Add the 'scripts' directory to the system path
import sys
sys.path.append("../scripts")

from functions import get_dataset_shape  

# Call functions to get an idea of dataset size
get_dataset_shape(fhvhv_df)

# Save original count of dataset
fhvhv_original_count = fhvhv_df.count()

Dataset shape is: 135975279 rows, 24 columns


In [4]:
# Display schema
fhvhv_df.printSchema()

# Preview a few records
fhvhv_df.show(7)

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [5]:
# Summary statistics
#sample_df = fhvhv_df.sample(fraction=0.05, seed=42)  # 5% of the data
#sample_df.describe().show()

In [6]:
# Add the 'scripts' directory to the system path
import sys
sys.path.append("../scripts")

from functions import get_missing_value_counts  

# Call function to inspect missing values
#get_missing_value_counts(fhvhv_df)

In [7]:
# Based on contextual knowledge, 
# drop feature columns deemed unuseful or unaligned with the research purpose

columns_to_drop = ['dispatching_base_num', 'DropOff_datetime', 'DOLocationID', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'base_passenger_fare', 'tolls', 
                   'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag']

fhvhv_df = fhvhv_df.drop(*columns_to_drop)
get_dataset_shape(fhvhv_df)
fhvhv_df.limit(7)


Dataset shape is: 135975279 rows, 7 columns


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75
HV0003,2023-05-01 00:06:54,121,3.88,810,1.0,12.71
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27
HV0003,2023-05-01 00:29:14,234,1.65,526,0.0,7.12
HV0003,2023-05-01 00:46:20,113,11.57,1886,0.0,32.93


In [8]:
from pyspark.sql import functions as F

# Filter out any invalid licenses that are not part of the big 4 rideshare companies
fhvhv_df = fhvhv_df.filter((F.col('hvfhs_license_num') == 'HV0003') | 
                (F.col('hvfhs_license_num') == 'HV0003') | 
                (F.col('hvfhs_license_num') == 'HV0003') | 
                (F.col('hvfhs_license_num') == 'HV0005'))

# Verify any changes to dataset size
get_dataset_shape(fhvhv_df)


Dataset shape is: 135975279 rows, 7 columns


In [9]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

# One hot encode this license feature
# Convert Strings in columns to a numeric index
indexer = StringIndexer(inputCol="hvfhs_license_num", outputCol="license_index")
fhvhv_df = indexer.fit(fhvhv_df).transform(fhvhv_df)

# Convert numeric indices to a one hot encoded vector
encoder = OneHotEncoder(inputCol="license_index", outputCol="license_vec")
fhvhv_df = encoder.fit(fhvhv_df).transform(fhvhv_df)

get_dataset_shape(fhvhv_df)
fhvhv_df.limit(7)

Dataset shape is: 135975279 rows, 9 columns


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay,license_index,license_vec
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27,1.0,"(1,[],[])"
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75,1.0,"(1,[],[])"
HV0003,2023-05-01 00:06:54,121,3.88,810,1.0,12.71,0.0,"(1,[0],[1.0])"
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57,0.0,"(1,[0],[1.0])"
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27,0.0,"(1,[0],[1.0])"
HV0003,2023-05-01 00:29:14,234,1.65,526,0.0,7.12,0.0,"(1,[0],[1.0])"
HV0003,2023-05-01 00:46:20,113,11.57,1886,0.0,32.93,0.0,"(1,[0],[1.0])"


In [10]:
from pyspark.sql.functions import col

# Ensure all dates are consistent and within the desired range
fhvhv_df = fhvhv_df.filter((col("pickup_datetime") >= "2023-05-01") & (col("pickup_datetime") < "2023-12-01"))
get_dataset_shape(fhvhv_df)


Dataset shape is: 135975279 rows, 9 columns


In [11]:
from pyspark.sql.functions import dayofweek, hour, month

# From pickup time variable, extract day of the week and hour information into separate column
fhvhv_df = fhvhv_df.withColumn("day_of_week", dayofweek("pickup_datetime"))
fhvhv_df = fhvhv_df.withColumn("hour_of_day", hour("pickup_datetime"))
fhvhv_df = fhvhv_df.withColumn("month", month("pickup_datetime"))

In [12]:
# One-hot encoding for 'month', 'day_of_week' and 'hour_of_day'

# For MONTH
#indexer_month = StringIndexer(inputCol="month", outputCol="month_index")
#fhvhv_df = indexer_month.fit(fhvhv_df).transform(fhvhv_df)

#encoder_month = OneHotEncoder(inputCol="month_index", outputCol="month_vec")
#fhvhv_df = encoder_month.fit(fhvhv_df).transform(fhvhv_df)

# For DAY
indexer_day = StringIndexer(inputCol="day_of_week", outputCol="day_index")
fhvhv_df = indexer_day.fit(fhvhv_df).transform(fhvhv_df)

encoder_day = OneHotEncoder(inputCol="day_index", outputCol="day_vec")
fhvhv_df = encoder_day.fit(fhvhv_df).transform(fhvhv_df)

# For HOUR
indexer_hour = StringIndexer(inputCol="hour_of_day", outputCol="hour_index")
fhvhv_df = indexer_hour.fit(fhvhv_df).transform(fhvhv_df)

encoder_hour = OneHotEncoder(inputCol="hour_index", outputCol="hour_vec")
fhvhv_df = encoder_hour.fit(fhvhv_df).transform(fhvhv_df)

get_dataset_shape(fhvhv_df)
fhvhv_df.limit(7)

Dataset shape is: 135975279 rows, 16 columns


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay,license_index,license_vec,day_of_week,hour_of_day,month,day_index,day_vec,hour_index,hour_vec
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])"
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])"
HV0003,2023-05-01 00:06:54,121,3.88,810,1.0,12.71,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])"
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])"
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])"
HV0003,2023-05-01 00:29:14,234,1.65,526,0.0,7.12,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])"
HV0003,2023-05-01 00:46:20,113,11.57,1886,0.0,32.93,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])"


In the preview above, May 1st 2023 is a Monday which is correctly represented as day_of_week = 2. (Since Sunday = 1, Monday = 2, etc)

In [13]:
from pyspark.sql.functions import date_trunc, col, to_date

# Truncate the pickup_datetime to the nearest hour to assist with joining the dataset later
fhvhv_df = fhvhv_df.withColumn("pickup_hour", date_trunc("hour", col("pickup_datetime")))

# Extract the date from the pickup_datetime to help with a join later as well
fhvhv_df = fhvhv_df.withColumn("pickup_date", to_date(col("pickup_datetime")))

# Preview
fhvhv_df.limit(7)


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay,license_index,license_vec,day_of_week,hour_of_day,month,day_index,day_vec,hour_index,hour_vec,pickup_hour,pickup_date
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01
HV0003,2023-05-01 00:06:54,121,3.88,810,1.0,12.71,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01
HV0003,2023-05-01 00:29:14,234,1.65,526,0.0,7.12,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01
HV0003,2023-05-01 00:46:20,113,11.57,1886,0.0,32.93,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01


In [14]:
# Ensure PULocationID is within the defined range
fhvhv_df = fhvhv_df.filter((F.col('PULocationID') >= 1) & (F.col('PULocationID')<= 263))
get_dataset_shape(fhvhv_df)

Dataset shape is: 135969214 rows, 18 columns


In [15]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

# One hot encode this PUlocationID feature
# Convert Strings in columns to a numeric index
indexer = StringIndexer(inputCol='PULocationID', outputCol="PULocation_index")
fhvhv_df = indexer.fit(fhvhv_df).transform(fhvhv_df)

# Convert numeric indices to a one hot encoded vector
encoder = OneHotEncoder(inputCol="PULocation_index", outputCol="PULocation_vec")
fhvhv_df = encoder.fit(fhvhv_df).transform(fhvhv_df)

get_dataset_shape(fhvhv_df)
fhvhv_df.limit(7)

Dataset shape is: 135969214 rows, 20 columns


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay,license_index,license_vec,day_of_week,hour_of_day,month,day_index,day_vec,hour_index,hour_vec,pickup_hour,pickup_date,PULocation_index,PULocation_vec
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,152.0,"(260,[152],[1.0])"
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,136.0,"(260,[136],[1.0])"
HV0003,2023-05-01 00:06:54,121,3.88,810,1.0,12.71,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,164.0,"(260,[164],[1.0])"
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,74.0,"(260,[74],[1.0])"
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,194.0,"(260,[194],[1.0])"
HV0003,2023-05-01 00:29:14,234,1.65,526,0.0,7.12,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,11.0,"(260,[11],[1.0])"
HV0003,2023-05-01 00:46:20,113,11.57,1886,0.0,32.93,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,55.0,"(260,[55],[1.0])"


In [16]:
# Add the 'scripts' directory to the system path
import sys
sys.path.append("../scripts")

from pyspark.sql import functions as F

from functions import apply_iqr_rule  


# Except for 'tips' which can be 0, filter the other columns for baseline
# (As it does not make sense for these columns to be 0)
fhvhv_df = fhvhv_df.filter((F.col('trip_miles') > 0.15)) # Assume a baseline distance people wont use Rideshare for
fhvhv_df = fhvhv_df.filter((F.col('trip_time') > 60)) # At least a minute worth of trip time
fhvhv_df = fhvhv_df.filter((F.col('tips') >= 0)) # Cannot have negative tips
fhvhv_df = fhvhv_df.filter((F.col('driver_pay') > 3)) # Assume a baseline pay

get_dataset_shape(fhvhv_df)

# Apply the rule to each relevant column
fhvhv_df = apply_iqr_rule(fhvhv_df, "trip_miles")
fhvhv_df = apply_iqr_rule(fhvhv_df, "trip_time")
fhvhv_df = apply_iqr_rule(fhvhv_df, "tips")
fhvhv_df = apply_iqr_rule(fhvhv_df, "driver_pay")

get_dataset_shape(fhvhv_df)


# Calculate min and max to inspect if outlier removal worked as expected
"""
fhvhv_df.select(
    F.min(F.col("trip_miles")).alias("min_trip_miles"),
    F.max(F.col("trip_miles")).alias("max_trip_miles"),
    F.min(F.col("trip_time")).alias("min_trip_time"),
    F.max(F.col("trip_time")).alias("max_trip_time"),
    F.min(F.col("tips")).alias("min_tips"),
    F.max(F.col("tips")).alias("max_tips"),
    F.min(F.col("driver_pay")).alias("min_driver_pay"),
    F.max(F.col("driver_pay")).alias("max_driver_pay")
).show()
"""

Dataset shape is: 135776633 rows, 20 columns


Dataset shape is: 106505700 rows, 20 columns


'\nfhvhv_df.select(\n    F.min(F.col("trip_miles")).alias("min_trip_miles"),\n    F.max(F.col("trip_miles")).alias("max_trip_miles"),\n    F.min(F.col("trip_time")).alias("min_trip_time"),\n    F.max(F.col("trip_time")).alias("max_trip_time"),\n    F.min(F.col("tips")).alias("min_tips"),\n    F.max(F.col("tips")).alias("max_tips"),\n    F.min(F.col("driver_pay")).alias("min_driver_pay"),\n    F.max(F.col("driver_pay")).alias("max_driver_pay")\n).show()\n'

In [17]:
# Add the 'scripts' directory to the system path
import sys
sys.path.append("../scripts")

from functions import standardise_column, get_dataset_shape

# Standardise the numeric feature columns 
columns_to_standardise = ["trip_miles", "trip_time"]
for column in columns_to_standardise:
    fhvhv_df = standardise_column(fhvhv_df, column)

get_dataset_shape(fhvhv_df)

# Preview
fhvhv_df.limit(7)

Dataset shape is: 106505700 rows, 22 columns


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay,license_index,license_vec,day_of_week,hour_of_day,month,day_index,day_vec,hour_index,hour_vec,pickup_hour,pickup_date,PULocation_index,PULocation_vec,trip_miles_standardised,trip_time_standardised
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,152.0,"(260,[152],[1.0])",-0.527518013859791,-0.6033874839120138
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,136.0,"(260,[136],[1.0])",-0.6252069902146785,-0.8592542144665771
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,74.0,"(260,[74],[1.0])",-0.5128994734551404,-0.5913937309172687
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,194.0,"(260,[194],[1.0])",1.1438684390719291,-0.08499082669469543
HV0003,2023-05-01 00:29:14,234,1.65,526,0.0,7.12,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,11.0,"(260,[11],[1.0])",-0.6660460872181468,-0.8152771201525115
HV0003,2023-05-01 00:46:20,113,11.57,1886,0.0,32.93,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,55.0,"(260,[55],[1.0])",1.6357939257046168,0.9971122212756452
HV0003,2023-05-01 00:00:59,243,0.9,323,0.0,5.38,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,70.0,"(260,[70],[1.0])",-0.8400763301306542,-1.0858028821450967


In [18]:
from pyspark.sql.functions import col, expr
from pyspark.sql.functions import round

# Calculate earnings AND earnings per hour (Possible Response Variables)
# 2 decimals places
# No division by zero because trip_time has already been filtered
fhvhv_df = fhvhv_df.withColumn("earnings_per_hour", round(((col("driver_pay") + col("tips")) / (col("trip_time") / 3600)), 2))
fhvhv_df = fhvhv_df.withColumn("earnings", round((col("driver_pay") + col("tips")), 2))

# Apply the rule again to the response variables, additional layer of outlier removal
fhvhv_df = apply_iqr_rule(fhvhv_df, "earnings")
fhvhv_df = apply_iqr_rule(fhvhv_df, "earnings_per_hour")

"""
# Calculate min and max to inspect if outlier removal worked as expected
fhvhv_df.select(
    F.min(F.col("earnings_per_hour")).alias("min_earnings_per_hour"),
    F.max(F.col("earnings_per_hour")).alias("max_earnings_per_hour"),
    F.min(F.col("earnings")).alias("min_earnings"),
    F.max(F.col("earnings")).alias("max_earnings")
).show()
"""

# Preview to check that it worked
fhvhv_df.limit(7)


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,tips,driver_pay,license_index,license_vec,day_of_week,hour_of_day,month,day_index,day_vec,hour_index,hour_vec,pickup_hour,pickup_date,PULocation_index,PULocation_vec,trip_miles_standardised,trip_time_standardised,earnings_per_hour,earnings
HV0005,2023-05-01 00:09:39,32,2.247,685,0.0,10.27,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,152.0,"(260,[152],[1.0])",-0.527518013859791,-0.6033874839120138,53.97,10.27
HV0005,2023-05-01 00:43:35,81,1.826,493,0.0,7.75,1.0,"(1,[],[])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,136.0,"(260,[136],[1.0])",-0.6252069902146785,-0.8592542144665771,56.59,7.75
HV0003,2023-05-01 00:24:27,92,2.31,694,0.0,10.57,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,74.0,"(260,[74],[1.0])",-0.5128994734551404,-0.5913937309172687,54.83,10.57
HV0003,2023-05-01 00:39:13,53,9.45,1074,0.0,24.27,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,194.0,"(260,[194],[1.0])",1.1438684390719291,-0.08499082669469543,81.35,24.27
HV0003,2023-05-01 00:29:14,234,1.65,526,0.0,7.12,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,11.0,"(260,[11],[1.0])",-0.6660460872181468,-0.8152771201525115,48.73,7.12
HV0003,2023-05-01 00:46:20,113,11.57,1886,0.0,32.93,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,55.0,"(260,[55],[1.0])",1.6357939257046168,0.9971122212756452,62.86,32.93
HV0003,2023-05-01 00:00:59,243,0.9,323,0.0,5.38,0.0,"(1,[0],[1.0])",2,0,5,6.0,"(6,[],[])",17.0,"(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,70.0,"(260,[70],[1.0])",-0.8400763301306542,-1.0858028821450967,59.96,5.38


In [19]:
 # Drop several feature columns no longer needed, part of the reponse variable/index

columns_to_drop = ['tips', 'driver_pay', 'license_index', 'day_index', 'hour_index', 'PULocation_index', ]

fhvhv_df = fhvhv_df.drop(*columns_to_drop)
get_dataset_shape(fhvhv_df)
fhvhv_df.limit(7)


Dataset shape is: 105043745 rows, 18 columns


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,license_vec,day_of_week,hour_of_day,month,day_vec,hour_vec,pickup_hour,pickup_date,PULocation_vec,trip_miles_standardised,trip_time_standardised,earnings_per_hour,earnings
HV0005,2023-05-01 00:09:39,32,2.247,685,"(1,[],[])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,"(260,[152],[1.0])",-0.527518013859791,-0.6033874839120138,53.97,10.27
HV0005,2023-05-01 00:43:35,81,1.826,493,"(1,[],[])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,"(260,[136],[1.0])",-0.6252069902146785,-0.8592542144665771,56.59,7.75
HV0003,2023-05-01 00:24:27,92,2.31,694,"(1,[0],[1.0])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,"(260,[74],[1.0])",-0.5128994734551404,-0.5913937309172687,54.83,10.57
HV0003,2023-05-01 00:39:13,53,9.45,1074,"(1,[0],[1.0])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,"(260,[194],[1.0])",1.1438684390719291,-0.08499082669469543,81.35,24.27
HV0003,2023-05-01 00:29:14,234,1.65,526,"(1,[0],[1.0])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,"(260,[11],[1.0])",-0.6660460872181468,-0.8152771201525115,48.73,7.12
HV0003,2023-05-01 00:46:20,113,11.57,1886,"(1,[0],[1.0])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,"(260,[55],[1.0])",1.6357939257046168,0.9971122212756452,62.86,32.93
HV0003,2023-05-01 00:00:59,243,0.9,323,"(1,[0],[1.0])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-01 00:00:00,2023-05-01,"(260,[70],[1.0])",-0.8400763301306542,-1.0858028821450967,59.96,5.38


In [20]:
# Evaluate how much of the original data is remaining after preprocessing.

Percentage_remaining = (fhvhv_df.count() / fhvhv_original_count)
print(Percentage_remaining)


0.7725209006557728


This next section will work on cleaning/pre-processing the external weather dataset

In [21]:
# Load weather data
weather_df = spark.read.csv('../data/raw_csv/NYC_hourly_weather_2023-05-01_to_2024-05-31.csv', header=True, inferSchema=True)


In [22]:
# Get an idea of dataset size
get_dataset_shape(weather_df)

# Display schema
weather_df.printSchema()

# Preview a few records
weather_df.show(7)

Dataset shape is: 9528 rows, 24 columns
root
 |-- name: string (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- temp: double (nullable = true)
 |-- feelslike: double (nullable = true)
 |-- dew: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- precip: double (nullable = true)
 |-- precipprob: integer (nullable = true)
 |-- preciptype: string (nullable = true)
 |-- snow: double (nullable = true)
 |-- snowdepth: double (nullable = true)
 |-- windgust: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- winddir: integer (nullable = true)
 |-- sealevelpressure: double (nullable = true)
 |-- cloudcover: double (nullable = true)
 |-- visibility: double (nullable = true)
 |-- solarradiation: integer (nullable = true)
 |-- solarenergy: double (nullable = true)
 |-- uvindex: integer (nullable = true)
 |-- severerisk: integer (nullable = true)
 |-- conditions: string (nullable = true)
 |-- icon: string (nullable = true)
 |-- stations: strin

In [23]:
# Summary statistics
weather_df.describe().show()


24/08/26 16:54:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------+-----------------+------------------+------------------+------------------+--------------------+-----------------+----------+--------------------+-------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+---------+--------------------+
|summary|         name|             temp|         feelslike|               dew|          humidity|              precip|       precipprob|preciptype|                snow|          snowdepth|          windgust|        windspeed|           winddir|  sealevelpressure|        cloudcover|        visibility|    solarradiation|       solarenergy|           uvindex|        severerisk|          conditions|     icon|            stations|
+-------+-------------+-----------------+------------------+------------------+------------------+--------------------+-----------------+-

In [24]:
# Add the 'scripts' directory to the system path
import sys
sys.path.append("../scripts")

from functions import get_missing_value_counts  
# Call function to inspect missing values
get_missing_value_counts(weather_df)

+----------+--------------+----------+---------------+---------+--------------+------------+----------------+----------------+----------+---------------+--------------+---------------+-------------+----------------------+----------------+----------------+--------------------+-----------------+-------------+----------------+----------------+----------+--------------+
|name_count|datetime_count|temp_count|feelslike_count|dew_count|humidity_count|precip_count|precipprob_count|preciptype_count|snow_count|snowdepth_count|windgust_count|windspeed_count|winddir_count|sealevelpressure_count|cloudcover_count|visibility_count|solarradiation_count|solarenergy_count|uvindex_count|severerisk_count|conditions_count|icon_count|stations_count|
+----------+--------------+----------+---------------+---------+--------------+------------+----------------+----------------+----------+---------------+--------------+---------------+-------------+----------------------+----------------+----------------+-------

In [25]:
# Since 'preciptype' is the only variable with missing values, inspect this more closely

# Find out how many distinct values the preciptype column can hold
unique_preciptypes = weather_df.select("preciptype").distinct()

# Show the distinct preciptypes
unique_preciptypes.show(truncate=False)

+----------+
|preciptype|
+----------+
|rain      |
|rain,snow |
|snow      |
|NULL      |
+----------+



In [26]:
# Make an ASSUMPTION here that NULL values represent the scenarios where there is no precipitation at all
# Logical assumpion as it cannot always be raining or snowing

from pyspark.sql.functions import when, col

# Replace NULL values in 'preciptype' with an empty string
weather_df = weather_df.withColumn("preciptype", when(col("preciptype").isNull(), "None").otherwise(col("preciptype")))

# Call function again to verify if this worked
get_missing_value_counts(weather_df)


+----------+--------------+----------+---------------+---------+--------------+------------+----------------+----------------+----------+---------------+--------------+---------------+-------------+----------------------+----------------+----------------+--------------------+-----------------+-------------+----------------+----------------+----------+--------------+
|name_count|datetime_count|temp_count|feelslike_count|dew_count|humidity_count|precip_count|precipprob_count|preciptype_count|snow_count|snowdepth_count|windgust_count|windspeed_count|winddir_count|sealevelpressure_count|cloudcover_count|visibility_count|solarradiation_count|solarenergy_count|uvindex_count|severerisk_count|conditions_count|icon_count|stations_count|
+----------+--------------+----------+---------------+---------+--------------+------------+----------------+----------------+----------+---------------+--------------+---------------+-------------+----------------------+----------------+----------------+-------

In [27]:
# Select only desired columns and drop everything else.
weather_df = weather_df.select(
    col('datetime'),
    col('feelslike'), 
    col('precip'), 
    col('preciptype')
)

# Preview
weather_df.limit(7)

datetime,feelslike,precip,preciptype
2023-05-01 00:00:00,60.4,0.004,rain
2023-05-01 01:00:00,59.0,0.002,rain
2023-05-01 02:00:00,56.2,0.004,rain
2023-05-01 03:00:00,52.9,0.005,rain
2023-05-01 04:00:00,50.8,0.0,None
2023-05-01 05:00:00,50.0,0.0,None
2023-05-01 06:00:00,44.5,0.0,None


In [28]:
from pyspark.sql.functions import col

# Ensure all dates are consistent and within the desired range
weather_df = weather_df.filter((col("datetime") >= "2023-05-01") & (col("datetime") < "2023-12-01"))
get_dataset_shape(weather_df)


Dataset shape is: 5137 rows, 4 columns


In [29]:
from pyspark.sql.functions import min, max

# Calculate min and max for 'feelslike'
weather_df.select(min("feelslike").alias("min_feelslike"), max("feelslike").alias("max_feelslike")).show()

# Calculate min and max for 'precip'
weather_df.select(min("precip").alias("min_precip"), max("precip").alias("max_precip")).show()


get_dataset_shape(weather_df)



+-------------+-------------+
|min_feelslike|max_feelslike|
+-------------+-------------+
|         19.2|        100.1|
+-------------+-------------+

+----------+----------+
|min_precip|max_precip|
+----------+----------+
|       0.0|     0.359|
+----------+----------+

Dataset shape is: 5137 rows, 4 columns


Using domain knowledge, I have decided not to use the IQR rule to remove any outliers from this dataset.

As the current min and max I have observed seem reasonable according to domain knowledge. 

For example, the coldest temperature in New York for 2023 was 3 degree Farenheit, so a min feelslike of 6.8 seems reasonable.

As weather data tends to have more outliers due to natural variability, I think leaving these reasonable values in can help better represent real world conditions.

Removing rows here could be problematic as it would mean the joined dataset later may have missing values which could cause problems with modelling.

In [30]:
# Add the 'scripts' directory to the system path
import sys
sys.path.append("../scripts")

from functions import standardise_column 

# Standardise the numeric feature columns 
columns_to_standardise = ["feelslike", "precip"]
for column in columns_to_standardise:
    weather_df = standardise_column(weather_df, column)

get_dataset_shape(weather_df)

# Preview
weather_df.limit(7)

Dataset shape is: 5137 rows, 6 columns


datetime,feelslike,precip,preciptype,feelslike_standardised,precip_standardised
2023-05-01 00:00:00,60.4,0.004,rain,-0.5020529657866335,0.31966025833655953
2023-05-01 01:00:00,59.0,0.002,rain,-0.6114140185896451,0.11452662347398555
2023-05-01 02:00:00,56.2,0.004,rain,-0.8301361241956681,0.31966025833655953
2023-05-01 03:00:00,52.9,0.005,rain,-1.08791574865991,0.42222707576784657
2023-05-01 04:00:00,50.8,0.0,None,-1.2519573278644276,-0.09060701138858843
2023-05-01 05:00:00,50.0,0.0,None,-1.314449358037577,-0.09060701138858843
2023-05-01 06:00:00,44.5,0.0,None,-1.7440820654779798,-0.09060701138858843


In [31]:
# This section works to one hot encode the 'preciptype' variable
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Convert strings to numeric index
indexer = StringIndexer(inputCol="preciptype", outputCol="preciptype_index")
weather_df = indexer.fit(weather_df).transform(weather_df)

# Convert index to one hot encoded vector
encoder = OneHotEncoder(inputCol="preciptype_index", outputCol="preciptype_vec")
weather_df = encoder.fit(weather_df).transform(weather_df)

# Preview
weather_df.limit(7)

datetime,feelslike,precip,preciptype,feelslike_standardised,precip_standardised,preciptype_index,preciptype_vec
2023-05-01 00:00:00,60.4,0.004,rain,-0.5020529657866335,0.31966025833655953,1.0,"(1,[],[])"
2023-05-01 01:00:00,59.0,0.002,rain,-0.6114140185896451,0.11452662347398555,1.0,"(1,[],[])"
2023-05-01 02:00:00,56.2,0.004,rain,-0.8301361241956681,0.31966025833655953,1.0,"(1,[],[])"
2023-05-01 03:00:00,52.9,0.005,rain,-1.08791574865991,0.42222707576784657,1.0,"(1,[],[])"
2023-05-01 04:00:00,50.8,0.0,None,-1.2519573278644276,-0.09060701138858843,0.0,"(1,[0],[1.0])"
2023-05-01 05:00:00,50.0,0.0,None,-1.314449358037577,-0.09060701138858843,0.0,"(1,[0],[1.0])"
2023-05-01 06:00:00,44.5,0.0,None,-1.7440820654779798,-0.09060701138858843,0.0,"(1,[0],[1.0])"


In [32]:
# Drop index feature column no longer needed

columns_to_drop = ['preciptype_index']

weather_df = weather_df.drop(*columns_to_drop)
get_dataset_shape(weather_df)
weather_df.limit(7)


Dataset shape is: 5137 rows, 7 columns


datetime,feelslike,precip,preciptype,feelslike_standardised,precip_standardised,preciptype_vec
2023-05-01 00:00:00,60.4,0.004,rain,-0.5020529657866335,0.31966025833655953,"(1,[],[])"
2023-05-01 01:00:00,59.0,0.002,rain,-0.6114140185896451,0.11452662347398555,"(1,[],[])"
2023-05-01 02:00:00,56.2,0.004,rain,-0.8301361241956681,0.31966025833655953,"(1,[],[])"
2023-05-01 03:00:00,52.9,0.005,rain,-1.08791574865991,0.42222707576784657,"(1,[],[])"
2023-05-01 04:00:00,50.8,0.0,None,-1.2519573278644276,-0.09060701138858843,"(1,[0],[1.0])"
2023-05-01 05:00:00,50.0,0.0,None,-1.314449358037577,-0.09060701138858843,"(1,[0],[1.0])"
2023-05-01 06:00:00,44.5,0.0,None,-1.7440820654779798,-0.09060701138858843,"(1,[0],[1.0])"


In [33]:
# Load public holidays data
holidays_2023_df = spark.read.csv('../data/raw_csv/2023_holidays.csv', header=True, inferSchema=True)
holidays_2024_df = spark.read.csv('../data/raw_csv/2024_holidays.csv', header=True, inferSchema=True)

# Combine the 2023 and 2024 holidays
holidays_df = holidays_2023_df.union(holidays_2024_df)

In [34]:
# Get an idea of dataset size
get_dataset_shape(holidays_df)

# Display schema
holidays_df.printSchema()

# Preview a few records
holidays_df.show(20)

Dataset shape is: 26 rows, 2 columns
root
 |-- Date: date (nullable = true)
 |-- Holiday: string (nullable = true)

+----------+--------------------+
|      Date|             Holiday|
+----------+--------------------+
|2023-01-01|      New Year's Day|
|2023-01-02|New Year's Day (O...|
|2023-01-16|Martin Luther Kin...|
|2023-02-20|     Presidents' Day|
|2023-05-29|        Memorial Day|
|2023-06-19|          Juneteenth|
|2023-07-04|    Independence Day|
|2023-09-04|           Labor Day|
|2023-10-09|        Columbus Day|
|2023-11-07|        Election Day|
|2023-11-10|Veteran's Day (Ob...|
|2023-11-11|       Veteran's Day|
|2023-11-23|    Thanksgiving Day|
|2023-12-25|       Christmas Day|
|2024-01-01|      New Year's Day|
|2024-01-15|Martin Luther Kin...|
|2024-02-19|     President's Day|
|2024-05-27|        Memorial Day|
|2024-06-19|          Juneteenth|
|2024-07-04|    Independence Day|
+----------+--------------------+
only showing top 20 rows



In [35]:
# Join the fhvhv dataset with the weather dataset
final_df = fhvhv_df.join(weather_df, fhvhv_df["pickup_hour"] == weather_df["datetime"], "left")

# Create a list of holiday dates
holiday_dates = [row['Date'] for row in holidays_df.select('Date').collect()] # extract as row objects and then iterate over to extract the date value

# Create an 'is_public_holiday' column, value 1 if the date is in the list, else 0
final_df = final_df.withColumn("is_public_holiday", col("pickup_date").isin(holiday_dates).cast("int"))

# Preview to check if the public_holiday indicator works/weather data joined
final_df.filter(F.col('is_public_holiday') == 1).limit(10)


hvfhs_license_num,pickup_datetime,PULocationID,trip_miles,trip_time,license_vec,day_of_week,hour_of_day,month,day_vec,hour_vec,pickup_hour,pickup_date,PULocation_vec,trip_miles_standardised,trip_time_standardised,earnings_per_hour,earnings,datetime,feelslike,precip,preciptype,feelslike_standardised,precip_standardised,preciptype_vec,is_public_holiday
HV0003,2023-05-29 00:07:46,100,7.09,1945,"(1,[0],[1.0])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[57],[1.0])",0.5962532747072395,1.075737935352308,55.3,29.88,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1
HV0003,2023-05-29 00:08:37,78,1.34,673,"(1,[0],[1.0])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[97],[1.0])",-0.7379785876219833,-0.619379154571674,50.87,9.51,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1
HV0005,2023-05-29 00:07:42,20,2.441,991,"(1,[],[])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[134],[1.0])",-0.4825021910264225,-0.19559988209067852,48.93,13.47,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1
HV0005,2023-05-29 00:29:18,47,0.722,389,"(1,[],[])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[117],[1.0])",-0.8813795077818892,-0.9978486935169656,50.62,5.47,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1
HV0005,2023-05-29 00:40:58,60,2.146,668,"(1,[],[])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[162],[1.0])",-0.5509540865720087,-0.6260423506798658,53.57,9.94,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1
HV0005,2023-05-29 00:57:30,94,3.061,1001,"(1,[],[])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[168],[1.0])",-0.3386371902187498,-0.18227348987429504,52.58,14.62,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1
HV0003,2023-05-29 00:21:31,117,1.11,331,"(1,[0],[1.0])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[185],[1.0])",-0.7913478621151521,-1.07514176837199,60.15,5.53,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1
HV0003,2023-05-29 00:32:03,117,7.99,1488,"(1,[0],[1.0])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[185],[1.0])",0.8050895662022485,0.4667218110635817,59.56,24.62,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1
HV0005,2023-05-29 00:27:18,65,0.918,440,"(1,[],[])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[76],[1.0])",-0.8358996043007539,-0.9298840932134097,44.75,5.47,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1
HV0005,2023-05-29 00:40:35,97,1.181,378,"(1,[],[])",2,0,5,"(6,[],[])","(23,[17],[1.0])",2023-05-29 00:00:00,2023-05-29,"(260,[58],[1.0])",-0.7748729991194347,-1.0125077249549874,52.1,5.47,2023-05-29 00:00:00,64.3,0.0,None,-0.1974043186925298,-0.09060701138858843,"(1,[0],[1.0])",1


In [36]:
# Drop feature columns not needed anymore (linking dataset columns)

columns_to_drop = ['pickup_hour', 'pickup_date', 'datetime', 'pickup_datetime', 'precip', 'preciptype', 'trip_time']

final_df = final_df.drop(*columns_to_drop)
get_dataset_shape(final_df)
final_df.limit(7)

final_df.printSchema()

Dataset shape is: 105090667 rows, 19 columns
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- license_vec: vector (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_vec: vector (nullable = true)
 |-- hour_vec: vector (nullable = true)
 |-- PULocation_vec: vector (nullable = true)
 |-- trip_miles_standardised: double (nullable = true)
 |-- trip_time_standardised: double (nullable = true)
 |-- earnings_per_hour: double (nullable = true)
 |-- earnings: double (nullable = true)
 |-- feelslike: double (nullable = true)
 |-- feelslike_standardised: double (nullable = true)
 |-- precip_standardised: double (nullable = true)
 |-- preciptype_vec: vector (nullable = true)
 |-- is_public_holiday: integer (nullable = true)



In [38]:
# Manually specify the distinct months
months = [5, 6, 7, 8, 9, 10, 11]

# List of months already processed
already_processed_months = [5, 6, 7, 8, 9, 10] 


# Exclude months that have already been processed
months_to_process = [month for month in months if month not in already_processed_months]

# Process and save each remaining chunk separately

for month_value in months_to_process:
    
    print(f"Processing chunk for month: {month_value}")
    chunk_df = final_df.filter(final_df.month == month_value)
    chunk_df.write.mode('append').parquet(f"../data/curated/chunk_{month_value}.parquet")
    print(f"Successfully saved chunk for month: {month_value}")


Processing chunk for month: 11


Successfully saved chunk for month: 11
